In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
class IncomeBracketParser(BaseEstimator, TransformerMixin):
    """
    Personalized Transformer to parse 'income_bracket' feature into a numeric format that can be used by ML models.
    It handles formats like '50K-100K', '100K+', and '50K' and converts them into a single numeric value representing the estimated income.
    - '50K-100K' -> 75000
    """
    def __init__(self, plus_multiplier=1.2):
        self.plus_multiplier = plus_multiplier

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        if 'income_bracket' in X_copy.columns:
            X_copy['income_numeric'] = X_copy['income_bracket'].apply(self._parse_income)
            X_copy = X_copy.drop(columns=['income_bracket'])
        return X_copy

    def _parse_income(self, val):
        if pd.isna(val): return 0
        val_str = str(val).strip()
        if '+' in val_str:
            base_val = float(val_str.replace('+', ''))
            return base_val * self.plus_multiplier
        elif '-' in val_str:
            parts = val_str.split('-')
            try: return (float(parts[0]) + float(parts[1])) / 2.0
            except ValueError: return 0
        else:
            try: return float(val_str)
            except ValueError: return 0

In [ ]:
print("Loading data...")
df = pd.read_csv('../data/product_purchase_dataset.csv')

X = df.drop(columns=['customer_id', 'purchase_amount'])
y = df['purchase_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
if 'income_bracket' in categorical_cols:
    categorical_cols.remove('income_bracket')

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.append('income_numeric')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

In [ ]:
pipeline = Pipeline(steps=[
    ('income_parser', IncomeBracketParser()),
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [ ]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

print("Initializing GridSearchCV (this may take a few minutes)...")
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1 
)

# Entrenamos usando el grid search
grid_search.fit(X_train, y_train)

# Extrajimos el mejor modelo que encontró
best_model = grid_search.best_estimator_

print("\n--- Optimization Results ---")
print(f"Best hyperparameters found:\n {grid_search.best_params_}")

In [ ]:
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n--- Metrics of the Best Model on Test Set ---")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"R^2 Score: {r2:.2f}")

In [ ]:
os.makedirs('../app/services/models', exist_ok=True)
joblib.dump(best_model, '../app/services/models/purchase_predictor.joblib')
print("\nModelo optimizado guardado exitosamente en '../app/services/models/purchase_predictor.joblib'")